In [1]:
import pandas as pd

file_path = "Wildfire_Dataset_61Day_Imputed.parquet"

df = pd.read_parquet(file_path, engine="pyarrow")

In [2]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import rowcol
from scipy import ndimage

df["datetime"] = pd.to_datetime(df["datetime"])

population_dir = Path("population_data")
population_col = "population"

year_to_tif = {}
for tif_path in sorted(population_dir.glob("*.tif")):
    match = re.search(r"usa_pop_(\d{4})_", tif_path.name)
    if match:
        year_to_tif[int(match.group(1))] = tif_path

assert year_to_tif, f"No population TIFFs found in {population_dir}"
available_population_years = np.array(sorted(year_to_tif), dtype=int)

df["population_year"] = df["datetime"].dt.year.astype(int)

def resolve_population_raster_year(year: int):
    if year in year_to_tif:
        return year
    if year >= 2014:
        nearest_idx = np.abs(available_population_years - year).argmin()
        return int(available_population_years[nearest_idx])
    return np.nan

df["population_raster_year"] = df["population_year"].map(resolve_population_raster_year)

# Sample once per unique (raster_year, lat, lon), then merge back to all rows.
coord_year_lookup = (
    df.loc[df["population_raster_year"].notna(), ["population_raster_year", "latitude", "longitude"]]
    .drop_duplicates()
    .copy()
)
coord_year_lookup["population_raster_year"] = coord_year_lookup["population_raster_year"].astype(int)
coord_year_lookup[population_col] = np.nan
filled_from_neighbor = 0

for raster_year, indexer in coord_year_lookup.groupby("population_raster_year", sort=True).groups.items():
    tif_path = year_to_tif[int(raster_year)]
    sample_rows = coord_year_lookup.loc[indexer, ["longitude", "latitude"]]

    with rasterio.open(tif_path) as src:
        assert str(src.crs) == "EPSG:4326", f"Unexpected CRS for {tif_path.name}: {src.crs}"
        band = src.read(1).astype(np.float32)
        nodata = src.nodata

        valid_mask = np.isfinite(band)
        if nodata is not None:
            valid_mask &= band != nodata
        assert valid_mask.any(), f"No valid population cells in {tif_path.name}"

        nearest_rows, nearest_cols = ndimage.distance_transform_edt(
            ~valid_mask,
            return_distances=False,
            return_indices=True,
        )
        filled_band = band[nearest_rows, nearest_cols]

        rows, cols = rowcol(
            src.transform,
            sample_rows["longitude"].to_numpy(),
            sample_rows["latitude"].to_numpy(),
        )
        rows = np.clip(np.asarray(rows), 0, src.height - 1)
        cols = np.clip(np.asarray(cols), 0, src.width - 1)

        exact_valid = valid_mask[rows, cols]
        filled_from_neighbor += int((~exact_valid).sum())
        samples = filled_band[rows, cols]

    coord_year_lookup.loc[indexer, population_col] = samples

df = df.merge(
    coord_year_lookup,
    on=["population_raster_year", "latitude", "longitude"],
    how="left",
    validate="many_to_one",
)

missing_2014_2026 = (
    df["population"].isna()
    & df["population_year"].between(2014, 2026)
).sum()
assert missing_2014_2026 == 0, f"Population still missing for {missing_2014_2026} rows in 2014-2026"
print(f"Filled {filled_from_neighbor} coordinate-year samples from nearest valid neighboring raster cells.")

df[["datetime", "latitude", "longitude", "population_year", "population_raster_year", population_col]].head()

df = df.drop(columns=["population_year", "population_raster_year"])

Filled 31743 coordinate-year samples from nearest valid neighboring raster cells.


In [3]:
cols_to_drop = [
    "candidate_window_id",
    "window_in_coord",
    "row_in_coord",
    "source_row"
]

df = df.drop(columns=cols_to_drop)

In [4]:
df = df.reset_index(names="_original_order")

# Keep each window together and sort chronologically inside the window.
df = df.sort_values(
    by=["window_id", "datetime", "_original_order"],
    kind="mergesort",
)

# Fill only within a window to avoid leaking values across different windows
# that happen to share the same latitude/longitude.
fill_cols = [col for col in df.columns if col not in ["window_id", "_original_order"]]
df[fill_cols] = (
    df.groupby("window_id", sort=False)[fill_cols]
    .transform(lambda group: group.ffill().bfill())
)

# Validate the window structure before restoring the incoming row order.
window_sizes = df.groupby("window_id").size()
assert window_sizes.nunique() == 1, window_sizes.value_counts().to_dict()
assert window_sizes.iloc[0] == 61, f"Expected 61 rows per window, found {window_sizes.iloc[0]}"
assert (df.groupby("window_id")["latitude"].nunique() == 1).all()
assert (df.groupby("window_id")["longitude"].nunique() == 1).all()

# Restore the original dataframe order after filling.
df = (
    df.sort_values(by="_original_order", kind="mergesort")
    .drop(columns="_original_order")
    .reset_index(drop=True)
)

In [5]:
import pandas as pd
import holidays
import numpy as np

df['datetime'] = pd.to_datetime(df['datetime'])

years = np.sort(df['datetime'].dt.year.unique())
us_holidays = holidays.US(years=years)
holiday_dates = np.sort(pd.to_datetime(list(us_holidays.keys())).values.astype('datetime64[D]'))
date_values = df['datetime'].values.astype('datetime64[D]')

# Find the nearest holiday for each row without using progress_apply.
insert_pos = np.searchsorted(holiday_dates, date_values)
prev_pos = np.clip(insert_pos - 1, 0, len(holiday_dates) - 1)
next_pos = np.clip(insert_pos, 0, len(holiday_dates) - 1)

prev_diff = np.abs((date_values - holiday_dates[prev_pos]).astype('timedelta64[D]').astype(int))
next_diff = np.abs((holiday_dates[next_pos] - date_values).astype('timedelta64[D]').astype(int))
df['prox_to_holiday'] = np.minimum(prev_diff, next_diff)

In [6]:
df.head()

,latitude,longitude,datetime,Wildfire,pr,rmax,rmin,sph,srad,tmmn,...,bi,fm100,fm1000,erc,etr,pet,vpd,window_id,population,prox_to_holiday
0,25.26027,-80.80099,2016-04-10,0,0.0,91.2,44.7,0.01111,306.6,290.7,...,37.0,16.4,18.6,27.0,8.0,5.9,0.71,25.2602699992712_-80.80099_0,2.488104e-14,50
1,25.26027,-80.80099,2016-04-11,0,0.0,100.0,51.0,0.01295,242.3,290.1,...,32.0,16.7,18.5,25.0,5.9,4.6,0.75,25.2602699992712_-80.80099_0,2.488104e-14,49
2,25.26027,-80.80099,2016-04-12,0,0.0,100.0,45.2,0.01282,299.7,289.6,...,27.0,16.7,18.4,27.0,6.6,5.3,0.79,25.2602699992712_-80.80099_0,2.488104e-14,48
3,25.26027,-80.80099,2016-04-13,0,0.0,100.0,47.0,0.01364,326.5,290.0,...,26.0,16.8,18.3,27.0,6.5,5.5,0.74,25.2602699992712_-80.80099_0,2.488104e-14,47
4,25.26027,-80.80099,2016-04-14,0,1.0,100.0,49.7,0.01437,292.7,292.0,...,27.0,16.9,18.2,27.0,6.4,5.2,0.78,25.2602699992712_-80.80099_0,2.488104e-14,46


In [7]:
df["datetime"] = pd.to_datetime(df["datetime"])
df["year"] = df["datetime"].dt.year

POP_COL = "population"

# Build one annual population series per location.
# Use median in case multiple rows exist for the same location-year.
annual_pop = (
    df.loc[df[POP_COL].notna(), ["latitude", "longitude", "year", POP_COL]]
    .groupby(["latitude", "longitude", "year"], as_index=False)[POP_COL]
    .median()
)

def predict_back_years(group: pd.DataFrame) -> pd.DataFrame:
    group = group.sort_values("year").copy()

    known = group[group[POP_COL].notna()]
    known = known[known["year"] >= 2015]

    if known.empty:
        return group

    x = known["year"].to_numpy(dtype=float)
    y = known[POP_COL].to_numpy(dtype=float)

    # If there is not enough variation/data for a fitted trend, use 2015 flat backfill.
    if len(np.unique(x)) < 2 or np.allclose(y, y[0]):
        group.loc[group["year"].isin([2013, 2014]) & group[POP_COL].isna(), POP_COL] = y[0]
        return group

    # Prefer log-linear growth if values are positive; otherwise use linear trend.
    if np.all(y > 0):
        coef = np.polyfit(x, np.log1p(y), 1)
        pred_fn = lambda yrs: np.expm1(np.polyval(coef, yrs))
    else:
        coef = np.polyfit(x, y, 1)
        pred_fn = lambda yrs: np.polyval(coef, yrs)

    target_mask = group["year"].isin([2013, 2014]) & group[POP_COL].isna()
    if target_mask.any():
        yrs = group.loc[target_mask, "year"].to_numpy(dtype=float)
        preds = pred_fn(yrs)
        preds = np.clip(preds, 0, None)
        group.loc[target_mask, POP_COL] = preds

    return group

# Create a location-year lookup including missing early years.
all_loc_years = (
    df[["latitude", "longitude", "year"]]
    .drop_duplicates()
    .merge(
        annual_pop,
        on=["latitude", "longitude", "year"],
        how="left",
    )
)

imputed_loc_years = (
    all_loc_years
    .groupby(["latitude", "longitude"], group_keys=False)
    .apply(predict_back_years)
    .reset_index(drop=True)
)

# Merge imputed annual population back to every row.
df = df.drop(columns=[POP_COL], errors="ignore").merge(
    imputed_loc_years,
    on=["latitude", "longitude", "year"],
    how="left",
    validate="many_to_one",
)

df = df.drop(columns=["year"])

/var/folders/5p/h0g4nrnj5fq_k9bbfh1drssh0000gn/T/ipykernel_62644/1327667625.py:62: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(predict_back_years)


In [8]:
df.head()

,latitude,longitude,datetime,Wildfire,pr,rmax,rmin,sph,srad,tmmn,...,bi,fm100,fm1000,erc,etr,pet,vpd,window_id,prox_to_holiday,population
0,25.26027,-80.80099,2016-04-10,0,0.0,91.2,44.7,0.01111,306.6,290.7,...,37.0,16.4,18.6,27.0,8.0,5.9,0.71,25.2602699992712_-80.80099_0,50,2.488104e-14
1,25.26027,-80.80099,2016-04-11,0,0.0,100.0,51.0,0.01295,242.3,290.1,...,32.0,16.7,18.5,25.0,5.9,4.6,0.75,25.2602699992712_-80.80099_0,49,2.488104e-14
2,25.26027,-80.80099,2016-04-12,0,0.0,100.0,45.2,0.01282,299.7,289.6,...,27.0,16.7,18.4,27.0,6.6,5.3,0.79,25.2602699992712_-80.80099_0,48,2.488104e-14
3,25.26027,-80.80099,2016-04-13,0,0.0,100.0,47.0,0.01364,326.5,290.0,...,26.0,16.8,18.3,27.0,6.5,5.5,0.74,25.2602699992712_-80.80099_0,47,2.488104e-14
4,25.26027,-80.80099,2016-04-14,0,1.0,100.0,49.7,0.01437,292.7,292.0,...,27.0,16.9,18.2,27.0,6.4,5.2,0.78,25.2602699992712_-80.80099_0,46,2.488104e-14


In [10]:
df["day_of_year"] = df["datetime"].dt.dayofyear
df["doy_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365)
df["doy_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365)

df = df.drop(columns=["day_of_year"])

In [11]:
df.columns

Index(['latitude', 'longitude', 'datetime', 'Wildfire', 'pr', 'rmax', 'rmin',
       'sph', 'srad', 'tmmn', 'tmmx', 'vs', 'bi', 'fm100', 'fm1000', 'erc',
       'etr', 'pet', 'vpd', 'window_id', 'prox_to_holiday', 'population',
       'doy_sin', 'doy_cos'],
      dtype='object')

In [12]:
df.to_parquet("Wildfire_Dataset_61Day_Holidays_n_doy_population.parquet")

In [13]:
df.head()

,latitude,longitude,datetime,Wildfire,pr,rmax,rmin,sph,srad,tmmn,...,fm1000,erc,etr,pet,vpd,window_id,prox_to_holiday,population,doy_sin,doy_cos
0,25.26027,-80.80099,2016-04-10,0,0.0,91.2,44.7,0.01111,306.6,290.7,...,18.6,27.0,8.0,5.9,0.71,25.2602699992712_-80.80099_0,50,2.488104e-14,0.985948,-0.167052
1,25.26027,-80.80099,2016-04-11,0,0.0,100.0,51.0,0.01295,242.3,290.1,...,18.5,25.0,5.9,4.6,0.75,25.2602699992712_-80.80099_0,49,2.488104e-14,0.982927,-0.183998
2,25.26027,-80.80099,2016-04-12,0,0.0,100.0,45.2,0.01282,299.7,289.6,...,18.4,27.0,6.6,5.3,0.79,25.2602699992712_-80.80099_0,48,2.488104e-14,0.979614,-0.200891
3,25.26027,-80.80099,2016-04-13,0,0.0,100.0,47.0,0.01364,326.5,290.0,...,18.3,27.0,6.5,5.5,0.74,25.2602699992712_-80.80099_0,47,2.488104e-14,0.976011,-0.217723
4,25.26027,-80.80099,2016-04-14,0,1.0,100.0,49.7,0.01437,292.7,292.0,...,18.2,27.0,6.4,5.2,0.78,25.2602699992712_-80.80099_0,46,2.488104e-14,0.972118,-0.234491


In [14]:
df.shape

(7685695, 24)

In [17]:
df["population"].describe()

count    7.685695e+06
mean     1.334397e+02
std      4.909011e+02
min      0.000000e+00
25%      8.008407e-02
50%      2.597173e+00
75%      2.983230e+01
max      8.863442e+03
Name: population, dtype: float64

In [ ]:
validation_df = df[["window_id", "latitude", "longitude", "datetime"]].copy()
validation_df["datetime"] = pd.to_datetime(validation_df["datetime"])

# Sort each window once so all downstream checks use the canonical ordering.
validation_df = validation_df.sort_values(
    ["window_id", "datetime"],
    kind="mergesort",
).reset_index(drop=True)

# Every window must contain exactly 61 consecutive daily rows at one coordinate.
window_sizes = validation_df.groupby("window_id", sort=False).size()
assert window_sizes.nunique() == 1, window_sizes.value_counts().to_dict()
assert window_sizes.iloc[0] == 61, f"Expected 61 rows per window, found {window_sizes.iloc[0]}"

lat_per_window = validation_df.groupby("window_id", sort=False)["latitude"].nunique()
lon_per_window = validation_df.groupby("window_id", sort=False)["longitude"].nunique()
assert (lat_per_window == 1).all(), "Some windows span multiple latitudes"
assert (lon_per_window == 1).all(), "Some windows span multiple longitudes"

window_day_steps = validation_df.groupby("window_id", sort=False)["datetime"].diff().dt.days.dropna()
assert (window_day_steps == 1).all(), window_day_steps.value_counts().to_dict()

# Exact coordinate-day rows must belong to only one window.
coord_day_membership = (
    validation_df
    .drop_duplicates(["latitude", "longitude", "datetime", "window_id"])
    .groupby(["latitude", "longitude", "datetime"], sort=False)["window_id"]
    .nunique()
)
assert coord_day_membership.max() == 1, (
    f"Found {(coord_day_membership > 1).sum()} coordinate-days shared across windows"
)

# Adjacent windows at the same coordinate must not overlap in calendar range.
window_bounds = validation_df.groupby("window_id", sort=False).agg(
    latitude=("latitude", "first"),
    longitude=("longitude", "first"),
    start=("datetime", "min"),
    end=("datetime", "max"),
)
window_bounds = window_bounds.sort_values(["latitude", "longitude", "start", "end"], kind="mergesort")
window_bounds["prev_end"] = window_bounds.groupby(["latitude", "longitude"], sort=False)["end"].shift()
overlap_mask = window_bounds["prev_end"].notna() & (window_bounds["start"] <= window_bounds["prev_end"])
assert not overlap_mask.any(), window_bounds.loc[overlap_mask, ["latitude", "longitude", "start", "end", "prev_end"]].head()

min_gap_days = (
    (window_bounds["start"] - window_bounds["prev_end"])
    .dt.days
    .dropna()
    .min()
)

print(
    "Window validation passed:",
    {
        "window_count": int(window_sizes.shape[0]),
        "rows_per_window": int(window_sizes.iloc[0]),
        "min_adjacent_window_gap_days": int(min_gap_days),
        "shared_coord_days": int((coord_day_membership > 1).sum()),
    },
)
